# 01 · Exploratory analysis

**Business question:** which headline wording raises clicks, and does it work differently for political and business news?

Everything here is exploration on the exploratory split. Patterns found here become hypotheses in `docs/PREREGISTRATION.md`, and are only believed if they hold in the confirmatory split (notebook 02).

In [ ]:
from headline_lab.checks import srm_by_test
from headline_lab.data import load_packages
from headline_lab.effects import effects_by_topic, topic_difference, within_test_effects
from headline_lab.features import FEATURES, add_features
from headline_lab.programme import (
    benjamini_hochberg,
    pvalues_by_test,
    share_with_no_effect,
    winners_curse,
)
from headline_lab.topics import tag_tests

packages = load_packages("../data/raw")
# drop tests that failed the split check (decided in notebook 00)
bad = srm_by_test(packages).query("has_mismatch")["clickability_test_id"]
packages = add_features(packages[~packages["clickability_test_id"].isin(bad)])
topics = tag_tests(packages)
features = list(FEATURES)

## 1. Which wording raises clicks? (all stories)
Effect per unit (per word, or yes vs no), as a share of the average click rate.

In [ ]:
effects = within_test_effects(packages, features)
effects[["feature", "effect_relative", "ci_low_relative", "ci_high_relative", "p_value"]].round(4)

**Takeaway:** TODO (Kaan)

## 2. Does it differ for politics and business?

In [ ]:
by_topic = effects_by_topic(packages, features, topics)
by_topic.pivot(index="feature", columns="topic", values="effect_relative").round(3)

In [ ]:
# formal test: does negativity work differently in political stories than in the rest?
for feature in ["negative_words", "outrage_words"]:
    for topic in ["politics", "business"]:
        diff = topic_difference(packages, feature, topics, topic)
        print(feature, topic, round(diff["effect_relative"], 3), "p =", round(diff["p_value"], 4))

**Takeaway:** TODO (Kaan)

## 3. The 2014 midterms
Did negativity in political headlines pay more in the two months before the election (4 November 2014)?

In [ ]:
created = packages.groupby("clickability_test_id")["created_at"].min()
in_window = created.between("2014-09-01", "2014-11-04")
season = in_window.map({True: "politics, election run-up", False: "other"})
season[topics != "politics"] = "not politics"
effects_by_topic(packages, ["negative_words", "outrage_words"], season).round(4)

**Takeaway:** TODO (Kaan). How many political tests fall in the window? Is that enough?

## 4. The programme: how many wins were real?

In [ ]:
p = pvalues_by_test(packages)
print("tests:", len(p))
print("significant at 0.05:", round((p < 0.05).mean(), 3))
print("still significant with FDR 5%:", round(benjamini_hochberg(p).mean(), 3))
print("estimated share with no real difference:", round(share_with_no_effect(p), 3))

In [ ]:
curse = winners_curse(packages)
curse, f"winners overrated by {curse.overstatement:.0%}"

**Takeaway:** TODO (Kaan)

## 5. What goes into the pre-registration
Pick 3–4 hypotheses from above, with the exact model, the direction you expect, and what result counts as "confirmed". Write them in `docs/PREREGISTRATION.md` and **commit before** opening notebook 02.